# Autoencoderでかなり見える異常検知演習

## Fashion-MNISTを使ったステップバイステップ実習

このノートブックでは、Autoencoderによる異常検知が比較的うまく働く例を扱います。

- 正常データ：**Sneaker（スニーカー）**
- 異常データ：**Ankle boot（ブーツ）**、Bag（バッグ）、Sandal（サンダル）など

Autoencoderは正常データだけを学習します。すると、正常画像はうまく復元できますが、形の違う異常画像は復元しにくくなります。
その結果、**再構成誤差**を使って異常を検出できます。

医用画像に進む前に、Autoencoder異常検知の基本原理を体験するための教材です。

## 第0段階：ColabのGPU確認

まず、GPUが使えるか確認します。
Colabのメニューで `ランタイム` → `ランタイムのタイプを変更` → `GPU` を選ぶと高速になります。

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

## 第1段階：Google Driveを接続する

学習したモデルや結果画像を保存するためにGoogle Driveを使います。
毎回Colabの環境はリセットされますが、Driveに保存しておけば後で再利用できます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/autoencoder_anomaly_fashionmnist'
os.makedirs(PROJECT_DIR, exist_ok=True)
print('保存先:', PROJECT_DIR)

## 第2段階：必要なライブラリを読み込む

今回は、画像データとしてFashion-MNISTを使います。
Fashion-MNISTは衣類や靴などの28×28画像データです。
医用画像ではありませんが、Autoencoder異常検知の原理を理解するには非常に扱いやすいデータです。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset

from torchvision import datasets, transforms

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support, confusion_matrix

# 結果をある程度再現しやすくする
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 第3段階：Fashion-MNISTをダウンロードする

`torchvision.datasets.FashionMNIST`を使うと、自動でデータをダウンロードできます。

ラベルは以下の10種類です。

| ラベル | 内容 |
|---:|---|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root='/content/data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='/content/data',
    train=False,
    download=True,
    transform=transform
)

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print('学習データ数:', len(train_dataset))
print('テストデータ数:', len(test_dataset))

## 第4段階：画像を表示してみる

まず、Fashion-MNISTの画像がどのようなものか確認します。
画像は28×28画素のグレースケール画像です。

In [ ]:
def show_samples(dataset, n=12):
    plt.figure(figsize=(12, 3))
    for i in range(n):
        x, y = dataset[i]
        plt.subplot(1, n, i + 1)
        plt.imshow(x.squeeze(), cmap='gray')
        plt.title(class_names[y], fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, n=12)

## 第5段階：今回の異常検知問題を作る

今回は次のように設定します。

- 正常：`Sneaker`、ラベル7
- 異常：`Ankle boot`、ラベル9

どちらも靴ですが、形が違います。
このくらいの違いだと、Autoencoder異常検知の効果がかなり見えやすくなります。

なお、より簡単にしたい場合は、異常をBagにするとさらに差が大きくなります。

In [ ]:
NORMAL_CLASS = 7   # Sneaker
ANOMALY_CLASS = 9  # Ankle boot

print('正常クラス:', NORMAL_CLASS, class_names[NORMAL_CLASS])
print('異常クラス:', ANOMALY_CLASS, class_names[ANOMALY_CLASS])

## 第6段階：正常画像だけで学習データを作る

Autoencoder異常検知では、原則として**正常データだけ**を使って学習します。

ここがCNN分類器との大きな違いです。

- CNN分類：正常画像と異常画像の両方を見て学習する
- Autoencoder異常検知：正常画像だけを見て、正常らしさを学習する

In [ ]:
# 学習データから正常クラスだけを取り出す
normal_train_indices = [i for i, (_, y) in enumerate(train_dataset) if y == NORMAL_CLASS]
normal_train_dataset = Subset(train_dataset, normal_train_indices)

print('正常画像だけの学習データ数:', len(normal_train_dataset))

BATCH_SIZE = 128
train_loader = DataLoader(normal_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

## 第7段階：評価用データを作る

評価では、正常画像と異常画像を混ぜます。
ここでは、テストデータからSneakerとAnkle bootだけを取り出します。

評価用ラベルは次のように作り直します。

- 正常：0
- 異常：1

この `0/1` ラベルは、異常検知の評価用です。

In [ ]:
test_images = []
test_labels = []
original_labels = []

for x, y in test_dataset:
    if y == NORMAL_CLASS:
        test_images.append(x)
        test_labels.append(0)  # 正常
        original_labels.append(y)
    elif y == ANOMALY_CLASS:
        test_images.append(x)
        test_labels.append(1)  # 異常
        original_labels.append(y)

test_images = torch.stack(test_images)
test_labels = torch.tensor(test_labels).long()
original_labels = torch.tensor(original_labels).long()

test_anomaly_dataset = TensorDataset(test_images, test_labels, original_labels)
test_loader = DataLoader(test_anomaly_dataset, batch_size=BATCH_SIZE, shuffle=False)

print('評価用画像数:', len(test_anomaly_dataset))
print('正常数:', int((test_labels == 0).sum()))
print('異常数:', int((test_labels == 1).sum()))

## 第8段階：正常画像と異常画像を見比べる

Autoencoderが学習するのは正常画像だけです。
まず正常画像と異常画像の見た目の違いを確認します。

In [ ]:
def show_normal_and_anomaly(test_images, test_labels, n=8):
    normal_idx = torch.where(test_labels == 0)[0][:n]
    anomaly_idx = torch.where(test_labels == 1)[0][:n]

    plt.figure(figsize=(12, 4))
    for i, idx in enumerate(normal_idx):
        plt.subplot(2, n, i + 1)
        plt.imshow(test_images[idx].squeeze(), cmap='gray')
        plt.title('normal\nSneaker', fontsize=9)
        plt.axis('off')

    for i, idx in enumerate(anomaly_idx):
        plt.subplot(2, n, n + i + 1)
        plt.imshow(test_images[idx].squeeze(), cmap='gray')
        plt.title('anomaly\nBoot', fontsize=9)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

show_normal_and_anomaly(test_images, test_labels, n=8)

## 第9段階：Autoencoderを定義する

Autoencoderは、入力画像をいったん小さな特徴表現に圧縮し、そこから元画像を復元するモデルです。

構造は次のようになります。

```text
入力画像
  ↓
Encoder：画像を圧縮する
  ↓
潜在表現
  ↓
Decoder：画像を復元する
  ↓
復元画像
```

今回は、理解しやすいように小さめの畳み込みAutoencoderを使います。

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder: 1×28×28 -> 16×14×14 -> 32×7×7
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )

        # Decoder: 32×7×7 -> 16×14×14 -> 1×28×28
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

model = ConvAutoencoder().to(device)
print(model)

## 第10段階：学習の設定をする

Autoencoderは、入力画像と復元画像の差が小さくなるように学習します。

ここでは損失関数としてMSEを使います。

MSEは、各画素の差の2乗平均です。

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15

## 第11段階：Autoencoderを学習する

ここで重要なのは、学習に使っているのは正常画像だけという点です。
異常画像は、学習中には一切見せていません。

In [ ]:
history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)

        optimizer.zero_grad()
        x_hat = model(x)
        loss = criterion(x_hat, x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    history.append(avg_loss)
    print(f'Epoch {epoch+1:02d}/{EPOCHS}  loss = {avg_loss:.6f}')

## 第12段階：学習曲線を見る

損失が下がっていれば、正常画像を復元する能力が上がっていることを意味します。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Autoencoder training loss')
plt.grid(True)
plt.show()

## 第13段階：正常画像の復元を確認する

Autoencoderが正常画像をどのくらい復元できているかを見ます。
上段が入力画像、下段が復元画像です。

In [ ]:
def show_reconstructions(model, images, title='reconstruction', n=8):
    model.eval()
    x = images[:n].to(device)
    with torch.no_grad():
        x_hat = model(x).cpu()

    plt.figure(figsize=(12, 3))
    for i in range(n):
        plt.subplot(2, n, i + 1)
        plt.imshow(images[i].squeeze(), cmap='gray')
        plt.title('input', fontsize=9)
        plt.axis('off')

        plt.subplot(2, n, n + i + 1)
        plt.imshow(x_hat[i].squeeze(), cmap='gray')
        plt.title('recon', fontsize=9)
        plt.axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

normal_images = test_images[test_labels == 0]
anomaly_images = test_images[test_labels == 1]

show_reconstructions(model, normal_images, title='Normal images: Sneaker', n=8)

## 第14段階：異常画像の復元を確認する

次に、学習していない異常画像を入れてみます。
AutoencoderはSneakerらしい画像だけを学習しているため、Bootをうまく復元できない場合があります。

In [ ]:
show_reconstructions(model, anomaly_images, title='Anomaly images: Ankle boot', n=8)

## 第15段階：再構成誤差を計算する

入力画像と復元画像の差を計算します。
この差が大きいほど、モデルにとって「見慣れない画像」である可能性が高くなります。

ここでは、1枚ごとにMSEを計算し、それを異常スコアとします。

In [ ]:
def compute_reconstruction_errors(model, loader):
    model.eval()
    all_errors = []
    all_labels = []
    all_original_labels = []

    with torch.no_grad():
        for x, y_anomaly, y_original in loader:
            x = x.to(device)
            x_hat = model(x)

            # 画像ごとのMSE: batch方向以外で平均
            errors = torch.mean((x - x_hat) ** 2, dim=(1, 2, 3))

            all_errors.append(errors.cpu())
            all_labels.append(y_anomaly)
            all_original_labels.append(y_original)

    all_errors = torch.cat(all_errors).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_original_labels = torch.cat(all_original_labels).numpy()

    return all_errors, all_labels, all_original_labels

errors, labels, orig_labels = compute_reconstruction_errors(model, test_loader)

print('再構成誤差の例:', errors[:10])
print('ラベルの例:', labels[:10])

## 第16段階：正常と異常の誤差分布を見る

ここがこの演習の重要ポイントです。
Autoencoderがうまく働いていれば、異常画像の再構成誤差が正常画像より大きくなるはずです。

In [ ]:
normal_errors = errors[labels == 0]
anomaly_errors = errors[labels == 1]

plt.figure(figsize=(8, 5))
plt.hist(normal_errors, bins=50, alpha=0.7, label='normal: Sneaker')
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='anomaly: Ankle boot')
plt.xlabel('Reconstruction error')
plt.ylabel('Count')
plt.title('Distribution of reconstruction errors')
plt.legend()
plt.grid(True)
plt.show()

print('正常画像の平均誤差:', normal_errors.mean())
print('異常画像の平均誤差:', anomaly_errors.mean())

## 第17段階：ROC曲線とAUCで評価する

異常スコアが高いほど異常とみなします。
ROC-AUCは、しきい値をいろいろ変えたときに、正常と異常をどれだけ分けられるかを表します。

- 0.5：ほぼランダム
- 1.0：完全に分離

この演習では、Autoencoderでもかなり高いAUCが出やすいはずです。

In [ ]:
auc = roc_auc_score(labels, errors)
fpr, tpr, thresholds = roc_curve(labels, errors)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curve')
plt.legend()
plt.grid(True)
plt.show()

print('ROC-AUC:', auc)

## 第18段階：しきい値を決める

異常スコアがしきい値より大きければ異常と判定します。

ここでは簡単に、正常画像の再構成誤差の95パーセンタイルをしきい値にします。
つまり、正常画像のうち約95%は正常と判定されるようにします。

In [ ]:
threshold = np.percentile(normal_errors, 95)
print('しきい値:', threshold)

pred = (errors > threshold).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(labels, pred, average='binary')
cm = confusion_matrix(labels, pred)

print('Precision:', precision)
print('Recall:', recall)
print('F1:', f1)
print('Confusion matrix:')
print(cm)

## 第19段階：混同行列を見やすく表示する

混同行列は、判定結果の内訳を表します。

| | 予測：正常 | 予測：異常 |
|---|---:|---:|
| 実際：正常 | TN | FP |
| 実際：異常 | FN | TP |

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(cm)
plt.xticks([0, 1], ['pred normal', 'pred anomaly'])
plt.yticks([0, 1], ['true normal', 'true anomaly'])
plt.title('Confusion matrix')

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=16)

plt.colorbar()
plt.tight_layout()
plt.show()

## 第20段階：誤差が小さい画像・大きい画像を見る

再構成誤差が小さい画像は、Autoencoderにとって復元しやすい画像です。
再構成誤差が大きい画像は、復元しにくい画像です。

ここでは、誤差が小さい順・大きい順に画像を確認します。

In [ ]:
def show_sorted_by_error(images, labels, errors, largest=True, n=10):
    idx = np.argsort(errors)
    if largest:
        idx = idx[::-1]
    idx = idx[:n]

    plt.figure(figsize=(12, 2.5))
    for i, k in enumerate(idx):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[k].squeeze(), cmap='gray')
        label_name = 'anomaly' if labels[k] == 1 else 'normal'
        plt.title(f'{label_name}\n{errors[k]:.4f}', fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

print('再構成誤差が大きい画像')
show_sorted_by_error(test_images, labels, errors, largest=True, n=10)

print('再構成誤差が小さい画像')
show_sorted_by_error(test_images, labels, errors, largest=False, n=10)

## 第21段階：誤差画像を可視化する

どの部分がうまく復元できていないかを見るために、入力画像と復元画像の差を表示します。

これは医用画像に進んだときの「異常部位の候補表示」の考え方につながります。

In [ ]:
def show_error_maps(model, images, labels, errors, target='largest', n=6):
    if target == 'largest':
        idx = np.argsort(errors)[::-1][:n]
    elif target == 'smallest':
        idx = np.argsort(errors)[:n]
    else:
        idx = np.arange(n)

    x = images[idx].to(device)
    model.eval()
    with torch.no_grad():
        x_hat = model(x).cpu()

    x_cpu = x.cpu()
    diff = torch.abs(x_cpu - x_hat)

    plt.figure(figsize=(12, 6))
    for i in range(n):
        # input
        plt.subplot(3, n, i + 1)
        plt.imshow(x_cpu[i].squeeze(), cmap='gray')
        plt.title('input', fontsize=8)
        plt.axis('off')

        # reconstruction
        plt.subplot(3, n, n + i + 1)
        plt.imshow(x_hat[i].squeeze(), cmap='gray')
        plt.title('recon', fontsize=8)
        plt.axis('off')

        # error map
        plt.subplot(3, n, 2*n + i + 1)
        plt.imshow(diff[i].squeeze(), cmap='hot')
        plt.title(f'error\n{errors[idx[i]]:.4f}', fontsize=8)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

show_error_maps(model, test_images, labels, errors, target='largest', n=6)

## 第22段階：モデルをGoogle Driveに保存する

学習済みモデルを保存しておくと、次回以降に再利用できます。

In [ ]:
model_path = os.path.join(PROJECT_DIR, 'conv_autoencoder_fashionmnist_sneaker.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'normal_class': NORMAL_CLASS,
    'anomaly_class': ANOMALY_CLASS,
    'threshold': float(threshold),
    'auc': float(auc),
}, model_path)

print('保存しました:', model_path)

## 第23段階：保存したモデルを読み込む

別の日に続きから実行する場合は、保存済みモデルを読み込めます。

In [ ]:
loaded = torch.load(model_path, map_location=device)

loaded_model = ConvAutoencoder().to(device)
loaded_model.load_state_dict(loaded['model_state_dict'])
loaded_model.eval()

loaded_threshold = loaded['threshold']
print('読み込んだしきい値:', loaded_threshold)
print('保存時のAUC:', loaded['auc'])

## 第24段階：1枚の画像を判定する関数を作る

最後に、1枚の画像について、

- 再構成誤差
- 正常／異常判定
- 入力画像
- 復元画像
- 誤差画像

を表示する関数を作ります。

In [ ]:
def predict_one_image(model, image, threshold):
    model.eval()
    x = image.unsqueeze(0).to(device)

    with torch.no_grad():
        x_hat = model(x)
        error = torch.mean((x - x_hat) ** 2).item()

    pred = 'anomaly' if error > threshold else 'normal'

    x_cpu = x.cpu().squeeze()
    x_hat_cpu = x_hat.cpu().squeeze()
    diff = torch.abs(x_cpu - x_hat_cpu)

    plt.figure(figsize=(8, 3))

    plt.subplot(1, 3, 1)
    plt.imshow(x_cpu, cmap='gray')
    plt.title('input')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(x_hat_cpu, cmap='gray')
    plt.title('reconstruction')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(diff, cmap='hot')
    plt.title('error map')
    plt.axis('off')

    plt.suptitle(f'prediction: {pred}   error={error:.5f}   threshold={threshold:.5f}')
    plt.tight_layout()
    plt.show()

    return error, pred

# 正常画像を1枚判定
error, pred = predict_one_image(loaded_model, normal_images[0], loaded_threshold)
print(error, pred)

# 異常画像を1枚判定
error, pred = predict_one_image(loaded_model, anomaly_images[0], loaded_threshold)
print(error, pred)

## 第25段階：発展課題

余裕がある場合は、以下を試してみてください。

### 課題1：正常クラスを変える

`NORMAL_CLASS = 7` を別のクラスに変えてみましょう。
たとえば、

- 正常：Bag
- 異常：Sneaker
- 正常：Trouser
- 異常：Dress

などにすると、難しさが変わります。

### 課題2：異常クラスを変える

`ANOMALY_CLASS = 9` を別のクラスに変えてみましょう。

正常Sneakerに対して、

- Sandalはやや近い
- Ankle bootは中くらい
- Bagはかなり違う

というように、異常の種類によって検出しやすさが変わります。

### 課題3：潜在表現を小さくする

EncoderやDecoderのチャネル数を減らすと、復元能力が下がります。
復元能力を下げると異常検知性能が上がることもありますが、正常画像まで復元できなくなることもあります。

### 課題4：医用画像に戻る

この演習でAutoencoder異常検知の基本を理解したあと、PneumoniaMNISTに戻ると、

> なぜ肺炎X線では再構成誤差だけで分けにくいのか

を考えやすくなります。

# まとめ

この演習では、Autoencoderによる異常検知の基本を確認しました。

重要な考え方は次の通りです。

1. 正常画像だけでAutoencoderを学習する。
2. 正常画像はうまく復元できる。
3. 異常画像は復元しにくい。
4. 入力画像と復元画像の差を異常スコアにする。
5. しきい値を決めて正常／異常を判定する。

このように、Autoencoderは「正常らしさから外れたもの」を見つける方法として理解できます。
ただし、医用画像のように異常が微妙で局所的な場合には、単純な再構成誤差だけでは十分に分離できないことがあります。